# FastAPI API Functionality Test

Learn and test the REST API layer in `src/api/app.py` and `src/api/middleware.py`.

Covered functionality:

- FastAPI app configuration
- request/response models
- rate limiter setup with Redis probe and `memory://` fallback
- `/health`
- `/query`
- `/query/stream` SSE response shape and `\n\n` event framing
- `/history/{thread_id}`
- `/agents/status`
- Teams router mounting

In [1]:
from pathlib import Path
import ast
import json
import sys

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'src' / 'api' / 'app.py').exists()), cwd)
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

api_path = project_root / 'src' / 'api' / 'app.py'
middleware_path = project_root / 'src' / 'api' / 'middleware.py'
source = api_path.read_text(encoding='utf-8', errors='replace')
middleware_source = middleware_path.read_text(encoding='utf-8', errors='replace')
lines = source.splitlines()
tree = ast.parse(source)

print('project_root:', project_root)
print('api_path:', api_path)
print('middleware_path:', middleware_path)
print('api_lines:', len(lines))

project_root: D:\0_PROJECTS\data-governance-copilot
api_path: D:\0_PROJECTS\data-governance-copilot\src\api\app.py
middleware_path: D:\0_PROJECTS\data-governance-copilot\src\api\middleware.py
api_lines: 164


In [2]:
def show_lines(start, end):
    end = min(end, len(lines))
    for n in range(start, end + 1):
        print(f'{n:4d}: {lines[n-1]}')

print('Top-level defs/classes:')
for node in tree.body:
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
        print(f'{node.lineno:4d}-{getattr(node, "end_lineno", node.lineno):4d}', type(node).__name__, node.name)

print('\nRoute decorators:')
for i, line in enumerate(lines, 1):
    if line.strip().startswith('@app.'):
        print(f'{i:4d}: {line.strip()}')

Top-level defs/classes:
  38-  44 ClassDef QueryRequest
  46-  57 ClassDef QueryResponse
  60-  74 AsyncFunctionDef _run_graph
  77-  78 AsyncFunctionDef health
  83- 102 AsyncFunctionDef query_endpoint
 106- 129 AsyncFunctionDef query_stream
 134- 142 AsyncFunctionDef history_endpoint
 147- 158 AsyncFunctionDef agents_status

Route decorators:
  76: @app.get("/health")
  81: @app.post("/query", response_model=QueryResponse)
 104: @app.post("/query/stream")
 132: @app.get("/history/{thread_id}")
 145: @app.get("/agents/status")


## 1. Request and Response Models

In [3]:
show_lines(34, 58)

assert 'class QueryRequest' in source
assert 'class QueryResponse' in source
assert 'approved' in source
assert 'pending_action' in source
print('PASS: API models include HITL fields')

  34:     allow_methods=["GET","POST"], allow_headers=["*"])
  35: 
  36: _executor = ThreadPoolExecutor(max_workers=int(os.getenv("MAX_WORKERS","4")))
  37: 
  38: class QueryRequest(BaseModel):
  39:     query:         str
  40:     thread_id:     Optional[str]       = None
  41:     user_id:       Optional[str]       = "api-user"
  42:     time_range:    Optional[str]       = "last_month"
  43:     data_products: Optional[List[str]] = []
  44:     approved:      Optional[bool]      = False   # Day 15 HITL flag
  45: 
  46: class QueryResponse(BaseModel):
  47:     query_id: str; 
  48:     thread_id: str; 
  49:     intent: str; 
  50:     summary: str
  51:     confidence: float; 
  52:     sources: List[str]; 
  53:     auto_tickets: List[str]
  54:     anomalies: List[str]; 
  55:     errors: List[dict]; 
  56:     execution_ms: float
  57:     pending_action: Optional[dict] = None
  58: 
PASS: API models include HITL fields


## 2. Rate Limiting Middleware

`limiter` uses IP-based limits for normal API clients. `user_limiter` keys Teams traffic by `X-User-Id` so one Teams IP does not throttle the entire org.

Current fix: `middleware.py` probes Redis at import time. If Redis is reachable, slowapi uses Redis storage; otherwise it falls back to `memory://` so tests and local dev do not crash when Redis is down.

In [5]:
#print(middleware_source)

from api.middleware import get_user_id, limiter, user_limiter, _redis_available, _storage_uri

class FakeRequest:
    def __init__(self, headers):
        self.headers = headers
        self.client = type('Client', (), {'host': '127.0.0.1'})()

assert get_user_id(FakeRequest({'X-User-Id': 'u123'})) == 'u123'
print('limiter:', limiter)
print('user_limiter:', user_limiter)
print('storage_uri:', _storage_uri)
assert _storage_uri == 'memory://' or _storage_uri.startswith('redis://')

limiter: <slowapi.extension.Limiter object at 0x000001BA1E678BF0>
user_limiter: <slowapi.extension.Limiter object at 0x000001BA1E708D40>
storage_uri: memory://


## 3. Import App and List Routes

If this fails, the missing dependency or graph/checkpointer issue is shown clearly.

In [6]:
api_import_error = None
try:
    from api.app import app, QueryRequest, QueryResponse
    print('app:', app)
    print('routes:')
    for route in app.routes:
        methods = sorted(getattr(route, 'methods', []) or [])
        print(methods, getattr(route, 'path', '?'))
except Exception as exc:
    api_import_error = exc
    print('API import failed:', repr(exc))

[nodes] InformationAgent not initialized: InformationAgent requires DATABRICKS_HOST, DATABRICKS_TOKEN, and DATABRICKS_HTTP_PATH to be set.
[nodes] KnowledgeAgent not initialized: KnowledgeAgent requires OPENAI_API_KEY for pgvector embeddings. Set OPENAI_API_KEY in your .env file.
[nodes] MetadataAgent not initialized: MetadataAgent requires COLLIBRA_BASE_URL and COLLIBRA_API_TOKEN environment variables.
[nodes] CapacityAgent not initialized: CapacityAgent requires JIRA_BASE_URL, JIRA_EMAIL, and JIRA_API_TOKEN environment variables.


PATH : data\memory.db
checkpointer : <langgraph.checkpoint.sqlite.SqliteSaver object at 0x000001BA2053AC00>
app: <fastapi.applications.FastAPI object at 0x000001BA1E6CBF80>
routes:
['GET', 'HEAD'] /openapi.json
['GET', 'HEAD'] /docs
['GET', 'HEAD'] /docs/oauth2-redirect
['GET', 'HEAD'] /redoc
['GET'] /health
['POST'] /query
['POST'] /query/stream
['GET'] /history/{thread_id}
['GET'] /agents/status
['POST'] /teams/webhook
['GET'] /teams/health


## 4. Model Validation Without Running The Server

In [7]:
if api_import_error is None:
    body = QueryRequest(query='Who owns the bookings dataset?', data_products=['bookings'], approved=False)
    print(body)
    response = QueryResponse(
        query_id='q1', thread_id='t1', intent='governance', summary='ok',
        confidence=0.9, sources=[], auto_tickets=[], anomalies=[], errors=[],
        execution_ms=123.0, pending_action=None,
    )
    print(response)
else:
    print('Skipped because api.app import failed.')

query='Who owns the bookings dataset?' thread_id=None user_id='api-user' time_range='last_month' data_products=['bookings'] approved=False
query_id='q1' thread_id='t1' intent='governance' summary='ok' confidence=0.9 sources=[] auto_tickets=[] anomalies=[] errors=[] execution_ms=123.0 pending_action=None


## 5. SSE Event Framing Check

Server-Sent Events must separate events with a blank line. The API should yield `data: ...\n\n`; without the terminator, clients can merge events.

In [8]:
sse_yields = [line.strip() for line in lines if line.strip().startswith('yield f"data:')]
for y in sse_yields:
    print(y)

assert sse_yields, 'No SSE yield statements found'
for y in sse_yields:
    assert '\\n\\n' in y, 'SSE yield is missing blank-line terminator: ' + y
print('PASS: SSE yield statements include \\n\\n terminators')

yield f"data: {json.dumps({'type':'start','query_id':qid})}\n\n"
yield f"data: {json.dumps(payload)}\n\n"
yield f"data: {json.dumps({'type':'done','execution_ms':result.get('execution_ms',0)})}\n\n"
yield f"data: {json.dumps({'type':'error','message':str(exc)})}\n\n"
PASS: SSE yield statements include \n\n terminators


## 6. Optional TestClient Smoke Test

This calls FastAPI in-process. It can trigger graph imports for query endpoints, so keep it small.

In [9]:
RUN_TESTCLIENT = False

if RUN_TESTCLIENT and api_import_error is None:
    from fastapi.testclient import TestClient
    client = TestClient(app)
    r = client.get('/health')
    print('GET /health:', r.status_code, r.json())
    assert r.status_code == 200

    r = client.get('/agents/status')
    print('GET /agents/status:', r.status_code, r.json())
else:
    print('Skipped. Set RUN_TESTCLIENT = True after api.app imports cleanly.')

Skipped. Set RUN_TESTCLIENT = True after api.app imports cleanly.
